In [1]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

documents = [file.parse() for file in reader.read()]

In [3]:
from embedder import Embedder
model = Embedder()
q1 = "How does approximate nearest neighbor search work?"
v1 = model.encode(q1)

In [4]:
v1[0]

np.float64(-0.02058203437252893)

In [9]:
import numpy as np
target = "02-vector-search/lessons/07-sqlitesearch-vector.md"

doc = next(
    (d for d in documents if d["filename"] == target),
    None
)

doc_text = doc['content']
doc_vec = model.encode(doc_text)

similarity = np.dot(v1, doc_vec)
print(similarity)

0.36107027225589694


In [10]:
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)

In [12]:
texts = [c["content"] for c in chunks]
X = model.encode_batch(texts)

In [13]:
query = "How does approximate nearest neighbor search work?"
v = model.encode(query)

scores = X.dot(v)

best_idx = np.argmax(scores)
chunks[best_idx]["filename"]

'02-vector-search/lessons/07-sqlitesearch-vector.md'

In [19]:
from minsearch import VectorSearch

texts = [doc["content"] for doc in documents]
vectors = model.encode_batch(texts)

index = VectorSearch()

index.fit(vectors, documents)

query = "What metric do we use to evaluate a search engine?"
qv = model.encode(query)

results = index.search(qv)

results[0]["filename"]

'04-evaluation/lessons/05-search-metrics.md'

In [20]:
from minsearch import Index

text_index = Index(
    text_fields=["content"]
)

text_index.fit(documents)

In [21]:
query = "How do I store vectors in PostgreSQL?"
qv = model.encode(query)

In [28]:
vector_index = VectorSearch()
vector_index.fit(vectors, documents)

In [33]:
text_results = text_index.search(query, num_results=5)
vector_results = vector_index.search(qv, num_results=5)

In [34]:
text_files = {r["filename"] for r in text_results}
vector_files = {r["filename"] for r in vector_results}

In [35]:
diff = vector_files - text_files
print(diff)

{'02-vector-search/lessons/04-vector-search.md', '02-vector-search/lessons/08-pgvector.md', '05-monitoring/lessons/05-database.md', '02-vector-search/lessons/07-sqlitesearch-vector.md'}
